# Experiment 1: Can Quantum Error Detection Protect a Magic State?

---

## Hypothesis

> **H1:** The $[\![4,2,2]\!]$ quantum error-detecting code can encode a
> single-qubit magic state $|T\rangle$ such that (a) the magic-state
> character is fully preserved, and (b) every single-qubit error is
> detectable by stabiliser measurement.

### Why this matters

Fault-tolerant quantum computing needs the $T$-gate, but the $T$-gate
cannot be implemented transversally on most error-correcting codes
(Eastin–Knill theorem). The workaround is to prepare a **magic state**
$|T\rangle = (|0\rangle + e^{i\pi/4}|1\rangle)/\sqrt{2}$ and consume
it via gate teleportation.

But a bare qubit has no error protection. If noise corrupts $|T\rangle$
before we use it, the entire computation is silently wrong. We need to
**encode** $|T\rangle$ into an error-detecting code so that corrupted
copies can be identified and discarded.

**The question:** Does the encoding actually work? Does it preserve the
magic, and can it catch errors?

### Claim

We claim that after encoding into the $[\![4,2,2]\!]$ code:
1. The magic witness $W = 1.0$ (perfect magic preserved).
2. Both stabiliser expectations are $+1$ (valid codeword).
3. Every single-qubit Pauli error ($X$, $Z$, $Y$) flips at least one
   stabiliser from $+1$ to $-1$.
4. Postselection on syndrome "00" correctly filters all detected errors.

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
from math import pi, sqrt

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp, state_fidelity
from qiskit.visualization import plot_bloch_multivector
from qiskit_aer import AerSimulator

from autoresearch_quantum.codes.four_two_two import (
    build_preparation_circuit, build_encoder, apply_magic_seed,
    encoded_magic_statevector, STABILIZERS, MEASUREMENT_OPERATORS, DATA_QUBITS,
)
from autoresearch_quantum.experiments.encoded_magic_state import build_circuit_bundle
from autoresearch_quantum.models import ExperimentSpec
from autoresearch_quantum.execution.analysis import logical_magic_witness

print("All imports successful.")

In [ ]:
from autoresearch_quantum.teaching import LearningTracker
from autoresearch_quantum.teaching.assess import quiz, predict_choice, reflect, order, checkpoint_summary
tracker = LearningTracker("plan_d_exp1")
print("Learning tracker active.")

---
## Part 1: The Magic State on a Single Qubit

Before we can test the encoding, we need to understand what we're
encoding. The magic state is:

$$|T\rangle = \frac{|0\rangle + e^{i\pi/4}|1\rangle}{\sqrt{2}}$$

It lives on the **equator** of the Bloch sphere, at $45°$ between the
$+X$ and $+Y$ axes. Its special property: it enables the $T$-gate via
gate teleportation — the key non-Clifford resource for universal quantum
computing.

In [ ]:
# Build the T-state
qc = QuantumCircuit(1, name="|T>")
qc.h(0)
qc.p(pi/4, 0)

t_state = Statevector.from_instruction(qc)
print("T-state amplitudes:")
print(f"  |0>: {t_state[0]:.4f}")
print(f"  |1>: {t_state[1]:.4f}")
print(f"  |1> phase: {np.angle(t_state[1])*180/pi:.1f} degrees = pi/4")

# Bloch coordinates
bloch = [t_state.expectation_value(SparsePauliOp(p)).real for p in ['X', 'Y', 'Z']]
print(f"\nBloch coordinates:")
print(f"  <X> = {bloch[0]:.4f}  (expected: 1/sqrt(2) = {1/sqrt(2):.4f})")
print(f"  <Y> = {bloch[1]:.4f}  (expected: 1/sqrt(2) = {1/sqrt(2):.4f})")
print(f"  <Z> = {bloch[2]:.4f}  (on the equator)")

In [ ]:
quiz(tracker, "q1_tstate_phase",
    question="What is the phase of the |1\u27E9 coefficient in the T-state?",
    options=["\u03C0/2 (90\u00b0)", "\u03C0/4 (45\u00b0)", "\u03C0/8 (22.5\u00b0)"],
    correct=1, section="1. T-state", bloom="remember",
    explanation="\u03C0/4 = 45\u00b0. The gate is called T (\u03C0/8 on the Bloch sphere), but the state phase is \u03C0/4.")

---
## Part 2: Encoding into the $[\![4,2,2]\!]$ Code

The $[\![4,2,2]\!]$ code uses **4 physical qubits** to encode **2 logical
qubits** with **distance 2** (detects any single-qubit error).

- **Logical qubit 0** ("the magic qubit"): will hold $|T\rangle$.
- **Logical qubit 1** ("the spectator"): stays in $|0\rangle_L$.

The codespace is the simultaneous $+1$ eigenspace of two stabilisers:
- $S_X = XXXX$
- $S_Z = ZZZZ$

Any state inside the codespace satisfies $\langle XXXX \rangle = +1$
and $\langle ZZZZ \rangle = +1$. An error kicks the state out of the
codespace, flipping at least one eigenvalue to $-1$.

In [ ]:
# Build the full preparation: seed (H+P) on qubit 0, then encode all 4
prep = build_preparation_circuit("h_p", "cx_chain")
print(f"Preparation circuit: {prep.num_qubits} qubits, depth {prep.depth()}")
prep.draw("mpl", style="iqp")

In [ ]:
# Compute the encoded statevector
state = encoded_magic_statevector()
print(f"Statevector has {len(state)} amplitudes (2^4 = 16)")
print(f"\nNon-zero amplitudes (the codespace):")
for i, amp in enumerate(state.data):
    if abs(amp) > 1e-10:
        print(f"  |{i:04b}> : {amp:.4f}  (magnitude: {abs(amp):.4f})")

In [ ]:
predict_choice(tracker, "q2_nonzero",
    question="How many of the 16 basis states have non-zero amplitude?",
    options=["2", "4", "8", "All 16"],
    correct=1, section="2. Encoding", bloom="understand",
    explanation="Only 4 basis states (0000, 0101, 1010, 1111) have non-zero amplitude. These span the codespace of the [[4,2,2]] code.")

---
## Part 3: Testing Claim (2) — Stabiliser Verification

**Claim:** Both stabiliser expectations are $+1$, confirming the
encoded state is a valid codeword.

In [ ]:
# Verify stabiliser expectations
state = encoded_magic_statevector()
for name, stab in STABILIZERS.items():
    exp = state.expectation_value(stab).real
    status = "PASS" if abs(exp - 1.0) < 1e-6 else "FAIL"
    print(f"  <{name}> = {exp:+.6f}  [{status}]")

**Result:** Both stabilisers read $+1$. The state is in the codespace. \checkmark

In [ ]:
quiz(tracker, "q3_stabilizer_meaning",
    question="\u27E8ZZZZ\u27E9 = +1 tells us:",
    options=[
        "All four qubits are in |0\u27E9",
        "The state is in the codespace \u2014 no X-type error detected",
        "The Z-gate has been applied to all qubits",
    ],
    correct=1, section="3. Stabilisers", bloom="understand",
    explanation="ZZZZ detects X errors (X anti-commutes with Z). Eigenvalue +1 means no X error is present.")

---
## Part 4: Testing Claim (3) — Every Single-Qubit Error Is Detectable

**Claim:** Every single-qubit Pauli error ($X$, $Z$, $Y$ on any of the
4 qubits) flips at least one stabiliser from $+1$ to $-1$.

We will systematically inject every possible single-qubit error and
check the stabilisers.

In [ ]:
# Complete error detection table
from qiskit.quantum_info import Operator
state = encoded_magic_statevector()

errors_detected = 0
errors_total = 0

header = f"{'Error':14s} {'<XXXX>':>8s} {'<ZZZZ>':>8s} {'Detected by':>15s}"
print(header)
print("=" * len(header))

for error_type in ['X', 'Y', 'Z']:
    for qubit in range(4):
        # Apply single-qubit error
        error_gate = {'X': np.array([[0,1],[1,0]]),
                      'Y': np.array([[0,-1j],[1j,0]]),
                      'Z': np.array([[1,0],[0,-1]])}[error_type]
        full_error = np.eye(1)
        for q in range(4):
            full_error = np.kron(full_error, error_gate if q == qubit else np.eye(2))
        corrupted = Statevector(full_error @ state.data)

        xxxx = corrupted.expectation_value(STABILIZERS["x_stabilizer"]).real
        zzzz = corrupted.expectation_value(STABILIZERS["z_stabilizer"]).real

        detected_by = []
        if abs(xxxx - (-1)) < 0.01: detected_by.append("XXXX")
        if abs(zzzz - (-1)) < 0.01: detected_by.append("ZZZZ")

        errors_total += 1
        if detected_by:
            errors_detected += 1

        det_str = ", ".join(detected_by) if detected_by else "NONE!"
        print(f"{error_type}(q{qubit}):       {xxxx:+.1f}     {zzzz:+.1f}     {det_str}")

print(f"\nDetected: {errors_detected}/{errors_total} single-qubit errors")

**Result:** All 12 single-qubit errors detected (12/12). \checkmark

- $X$ errors: detected by $ZZZZ$ (because $X$ anti-commutes with $Z$)
- $Z$ errors: detected by $XXXX$ (because $Z$ anti-commutes with $X$)
- $Y$ errors: detected by **both** (because $Y = iXZ$)

In [ ]:
quiz(tracker, "q4_which_detects",
    question="A Z error on qubit 2 occurs. Which stabiliser detects it?",
    options=[
        "ZZZZ (because Z commutes with Z \u2014 wait, that means it does NOT detect it)",
        "XXXX (because Z anti-commutes with X, flipping the eigenvalue)",
        "Neither \u2014 Z errors are invisible",
    ],
    correct=1, section="4. Error detection", bloom="apply",
    explanation="Z anti-commutes with X. A Z error on any qubit flips \u27E8XXXX\u27E9 from +1 to \u22121.")

In [ ]:
order(tracker, "q5_error_severity",
    instruction="Rank error types by how many stabilisers they trigger (fewest \u2192 most):",
    items=["X", "Z", "Y"],
    correct_order=["X", "Z", "Y"],
    section="4. Error detection", bloom="analyze",
    explanation="X \u2192 1 (ZZZZ). Z \u2192 1 (XXXX). Y \u2192 2 (both). X and Z are tied at 1.",
    ties=[["X", "Z"]])

---
## Part 5: Testing Claim (1) — The Magic Witness

**Claim:** The magic witness $W = 1.0$, proving the encoded state fully
preserves the $T$-state character.

The witness formula:
$$W = \frac{1 + \frac{\langle X_L \rangle + \langle Y_L \rangle}{\sqrt{2}}}{2}
\times \frac{1 + \langle Z_{\text{spec}} \rangle}{2}$$

In [ ]:
# Measure logical operators
state = encoded_magic_statevector()
results = {}
for name, op_dict in MEASUREMENT_OPERATORS.items():
    pauli_str = ["I"] * 4
    for qubit, basis in op_dict.items():
        pauli_str[qubit] = basis
    label = "".join(reversed(pauli_str))
    op = SparsePauliOp(label)
    results[name] = state.expectation_value(op).real

lx, ly, sz = results["logical_x"], results["logical_y"], results["spectator_z"]
print(f"<X_L>          = {lx:+.6f}   (ideal: +1/sqrt(2) = +{1/sqrt(2):.6f})")
print(f"<Y_L>          = {ly:+.6f}   (ideal: +1/sqrt(2) = +{1/sqrt(2):.6f})")
print(f"<Z_spectator>  = {sz:+.6f}   (ideal: +1.000000)")

magic_factor = (1 + (lx + ly)/sqrt(2)) / 2
spec_factor = (1 + sz) / 2
W = magic_factor * spec_factor

print(f"\nMagic factor     = {magic_factor:.6f}")
print(f"Spectator factor = {spec_factor:.6f}")
print(f"Witness W        = {W:.6f}")
print(f"Library check    = {logical_magic_witness(lx, ly, sz):.6f}")

**Result:** $W = 1.0$. The encoding perfectly preserves the magic-state character. \checkmark

In [ ]:
quiz(tracker, "q6_ideal_witness",
    question="For a perfect T-state, the magic witness W equals:",
    options=["0.0", "0.5", "1/\u221A2 \u2248 0.707", "1.0"],
    correct=3, section="5. Witness", bloom="apply",
    explanation="Ideal: magic_factor = 1.0, spectator_factor = 1.0. Product = 1.0.")

---
## Part 6: Testing Claim (4) — Postselection Works

**Claim:** Syndrome-based postselection correctly identifies all
detected errors. On an ideal simulator, 100% of shots have syndrome "00"
(no error detected).

In [ ]:
# Build the full circuit bundle and run on ideal simulator
spec = ExperimentSpec(rung=1, seed_style="h_p", encoder_style="cx_chain",
                      verification="both", postselection="all_measured",
                      shots=512, repeats=1)
bundle = build_circuit_bundle(spec)

sim = AerSimulator()
from autoresearch_quantum.execution.analysis import summarize_context, local_memory_records

total_accepted = 0
total_shots = 0
for name, circ in bundle.witness_circuits.items():
    job = sim.run(circ, shots=512, memory=True)
    memory = job.result().get_memory()
    records = local_memory_records(memory, [cr.name for cr in circ.cregs])
    summary = summarize_context(records, ["z_stabilizer", "x_stabilizer"],
                                spec.postselection, MEASUREMENT_OPERATORS[name])
    total_accepted += summary["accepted_shots"]
    total_shots += summary["total_shots"]
    print(f"{name:15s}: acceptance = {summary['acceptance_rate']:.4f}, "
          f"<operator> = {summary['expectation']:+.4f}")

print(f"\nOverall acceptance: {total_accepted}/{total_shots} "
      f"= {total_accepted/total_shots:.4f}")

**Result:** 100% acceptance on the ideal simulator. Every shot has syndrome "00". \checkmark

In [ ]:
quiz(tracker, "q7_acceptance_ideal",
    question="On an ideal simulator, what fraction of shots pass the syndrome check?",
    options=["About 50%", "About 75%", "100%"],
    correct=2, section="6. Postselection", bloom="understand",
    explanation="No noise means no errors. Every shot is in the codespace, so every syndrome is 00.")

---
## Proof Summary

| Claim | Result | Status |
|-------|--------|--------|
| (1) Magic witness $W = 1.0$ | $W = 1.000000$ | **Proven** |
| (2) Both stabilisers at $+1$ | $\langle XXXX \rangle = +1$, $\langle ZZZZ \rangle = +1$ | **Proven** |
| (3) Every 1-qubit error detected | 12/12 detected | **Proven** |
| (4) Postselection filters correctly | 100% acceptance (ideal) | **Proven** |

**Hypothesis H1 is confirmed.** The $[\![4,2,2]\!]$ code can encode a
magic state with perfect fidelity, and its error detection works exactly
as the theory predicts.

---

## But Wait — Next Hypothesis

> **H2 (for Experiment 2):** Everything above was on a **perfect
> simulator** with zero noise. On a realistic noise model (mimicking
> IBM Brisbane, 127 qubits, real error rates), the magic-state quality
> will degrade — but the degradation is **quantifiable**, and by tuning
> circuit parameters we can recover significantly more magic than a
> naive default configuration.

**The question Experiment 2 will answer:** How much magic survives
real-world noise, and can we measure the damage precisely enough to
optimise against it?

In [ ]:
checkpoint_summary(tracker, "6. Postselection")

---
## Assessment

In [ ]:
tracker.dashboard()
path = tracker.save()
print(f"\nProgress saved to: {path}")